# Modul 0 · Deine Entwicklungsumgebung
**Advanced Data Science · Advanced AI Engineering** · linkit courses

Seit Januar 2025 gehört das Werk Illinois bei Chicago zu Nordwerk. Beide Werke schicken ihre Schichtmeldungen an die Datenanalyse, jedes in seinem Format: das Stammwerk eine Datei je Produktionstag mit Semikolon, Dezimalkomma und deutschem Datum, Illinois eine Datei je Woche mit Komma, US-Datum und Temperaturen in Fahrenheit. Bisher hat jemand die Dateien von Hand in Excel zusammenkopiert. In diesem Notebook baust du den Importer, der das übernimmt: Er findet die Lieferungen, liest jede Kodierung richtig, prüft jede Zeile, führt beide Werke in einer Tabelle zusammen und schreibt einen Qualitätsbericht. Am Ende steht eine Prüfsumme, die auf jedem Rechner gleich ist. Danach wandern die Funktionen als Paket `nordwerk` mit dem Befehl `nordwerk import` in dein eigenes Repository.

**Lernziele**
1. Schichten einer Umgebung benennen und typische Brüche erkennen
2. Python-Version, virtuelle Umgebung und Abhängigkeiten mit uv verwalten
3. Versionsangabe und Lockfile unterscheiden und beide pflegen
4. Ein Paket im src-Layout mit eigenem Kommandozeilenbefehl bauen
5. Dateien mit pathlib, richtiger Kodierung, Ausnahmen und Logging lesen
6. Generatoren, Scope, Typannotationen und dataclasses gezielt einsetzen
7. Konfiguration, Geheimnisse, IDE und Debugger sicher nutzen

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Der Hauptweg ist dein Laptop: Lade das Notebook herunter, leg es in deinem Projekt unter `notebooks/` ab und öffne es in VS Code mit dem Kernel aus deiner `.venv` (Schritt „Auf deinem Laptop“ der Kursseite, Skript Abschnitt 7.4). Colab geht auch, ist aber nicht nötig.
- Das Notebook arbeitet in seinem eigenen Ordner. In VS Code liegen die Daten deshalb unter `notebooks/daten/`, relativ zum Notebook. Alle Pfade hier sind relativ (`Path("daten")`), nie absolut. Die Zeile `daten/` in deiner `.gitignore` hält auch diesen Ordner aus dem Repository.
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus. Du brauchst nur die Standardbibliothek von Python 3.13.
- Wo `raise NotImplementedError("TODO 4")` steht, schreibst du deinen Code. Darunter prüft ein Selbsttest dein Ergebnis: ✅ stimmt, ❌ stimmt noch nicht (mit Hinweis), ⏳ wartet auf ein früheres TODO, das noch offen ist oder dessen Selbsttest noch kein ✅ zeigt. Das Notebook läuft in jedem Fall weiter, auch mit „Alle ausführen“.
- Zellen, die mit `# → src/nordwerk/importer.py` (oder `konfig.py`, `cli.py`) beginnen, gehören ins Paket. Schreib dort nur Definitionen, keine Ausgaben und keine Tests. Dann übernimmst du sie unverändert in dein Repository (Abschnitt „Ins Repo“ am Ende).
- Mindestziel sind TODO 1 bis 12. Die beiden Bonusaufgaben vertiefen.
- Jeder Teil gehört zu einem Abschnitt des Skripts. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfen für die Selbsttests: einfach ausführen, nicht ändern.
import contextlib
import datetime as dt
import inspect
import io
import json
import logging
import os
import re
import tempfile
from dataclasses import FrozenInstanceError, fields, is_dataclass
from pathlib import Path


GRUEN = {}


def check(name, test):
    """Führt einen Test aus: ✅ stimmt, ❌ stimmt noch nicht, ⏳ ein früheres TODO ist noch offen."""
    eigenes = re.match(r"TODO \d+|Bonus \d", name).group(0)
    GRUEN[name] = False
    try:
        test()
    except NotImplementedError as e:
        if str(e) and str(e) != eigenes:
            print(f"⏳ {name}: wartet auf {e}")
        else:
            print(f"❌ {name}: noch nicht gelöst")
    except AssertionError as e:
        print(f"❌ {name}: {e}")
    except SystemExit as e:
        print(f"❌ {name}: Das Programm wurde mit SystemExit({e.code}) beendet. main soll den Exit-Code mit return zurückgeben.")
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        GRUEN[name] = True
        print(f"✅ {name}")


def braucht(*labels):
    """Ein Test, der frühere TODOs nutzt, wartet, bis deren Selbsttests alle ✅ zeigen."""
    for label in labels:
        eigene = [ok for name, ok in GRUEN.items() if re.match(r"TODO \d+|Bonus \d", name).group(0) == label]
        if not eigene or not all(eigene):
            raise NotImplementedError(label)


def zeige(was):
    """Zeigt ein Zwischenergebnis. Solange ein nötiges TODO offen ist, steht dort nur ein Hinweis."""
    try:
        wert = was()
    except NotImplementedError as e:
        print(f"(noch offen: {e})")
    except Exception as e:
        print(f"(Fehler: {type(e).__name__}: {e})")
    except SystemExit as e:
        print(f"(SystemExit({e.code}): Das Programm wurde beendet. main soll den Exit-Code mit return zurückgeben.)")
    else:
        if wert is not None:
            print(wert)


class _Sammler(logging.Handler):
    def __init__(self):
        super().__init__(logging.DEBUG)
        self.meldungen = []

    def emit(self, record):
        self.meldungen.append((record.levelname, record.name, record.getMessage()))


@contextlib.contextmanager
def logger_gesichert():
    """Stellt den Logger "nordwerk" nach einem Test wieder so her, wie er vorher war."""
    logger = logging.getLogger("nordwerk")
    vorher = (logger.level, logger.propagate, logger.handlers[:])
    try:
        yield logger
    finally:
        logger.setLevel(vorher[0])
        logger.propagate = vorher[1]
        logger.handlers[:] = vorher[2]


def mitschreiben(arbeit):
    """Führt arbeit() aus und sammelt die Meldungen aller Logger unter "nordwerk", ohne sie anzuzeigen."""
    sammler = _Sammler()
    with logger_gesichert() as logger:
        logger.handlers[:] = [sammler]
        logger.setLevel(logging.DEBUG)
        logger.propagate = False
        ergebnis = arbeit()
    return ergebnis, sammler.meldungen


print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegt das Zip schon bereit, in VS Code und Colab holt diese Zelle es von GitHub.
import os, urllib.request, zipfile
DATEN = ["lieferungen.zip"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/advanced/daten/"

# Alle Pfade sind relativ zum Ordner dieses Notebooks: in VS Code also notebooks/daten/, im Browser daten/.
DATENORDNER = Path("daten")
EINGANG = DATENORDNER / "roh"
DATENORDNER.mkdir(exist_ok=True)
for datei in DATEN:
    ziel = DATENORDNER / datei
    if not ziel.exists():
        teil = ziel.with_name(ziel.name + ".teil")  # erst vollständig laden, dann umbenennen
        urllib.request.urlretrieve(QUELLE + datei, teil)
        teil.replace(ziel)
if not EINGANG.is_dir():
    try:
        with zipfile.ZipFile(DATENORDNER / "lieferungen.zip") as archiv:
            archiv.extractall(DATENORDNER)
    except zipfile.BadZipFile:
        (DATENORDNER / "lieferungen.zip").unlink()
        raise RuntimeError("Das Zip war beschädigt und ist gelöscht. Führ diese Zelle noch einmal aus.") from None
print("Lieferungen bereit:", len(list(EINGANG.iterdir())), "Dateien in", EINGANG)

Neben den Lieferungen liegt `liesmich.txt` von der Werks-IT. Sie beschreibt, was beide Werke schicken; lies sie einmal ganz.

In [ ]:
print((EINGANG / "liesmich.txt").read_text(encoding="utf-8"))

## Vorgaben
Die nächste Zelle legt fest, was beide Werke liefern. Sie ist die erste Zelle von `importer.py`, deshalb stehen dort alle Importe des Moduls.

- `LieferungsFehler`: Die ganze Datei ist unbrauchbar (leer, Pflichtspalte fehlt). Der Import lehnt sie ab und macht mit der nächsten weiter.
- `ZeilenFehler(grund, text)`: Eine Zeile ist unbrauchbar. `grund` ist einer aus `GRUENDE` oder `"summenzeile"`. Weil `ZeilenFehler` von `ValueError` erbt, passt `except ValueError` auch.
- `Format` und `FORMATE`: je Werk das Dateimuster, das Trennzeichen, das Datumsformat, das Dezimalzeichen, ob die Temperatur in Fahrenheit kommt, die Spalten (Name in der Lieferung → Feld) und die Schichten (Name in der Lieferung → `Früh`, `Spät` oder `Nacht`).
- `SCHICHT_RANG` ordnet die Schichten im Tagesablauf, `PFLICHT` nennt die Felder, die nie leer sein dürfen.
- `log` ist der Logger des Moduls. Bibliothekscode meldet über ihn, nie mit `print` (Skript Abschnitt 4.5).

In [ ]:
# → src/nordwerk/importer.py
"""Lesen, Prüfen, Zusammenführen und Schreiben der Schichtmeldungen beider Nordwerk-Werke."""

import codecs
import csv
import datetime as dt
import hashlib
import io
import logging
from collections.abc import Iterator
from dataclasses import dataclass, fields
from pathlib import Path
from typing import Any

log = logging.getLogger("nordwerk.importer")


class LieferungsFehler(Exception):
    """Die ganze Datei ist unbrauchbar, etwa leer oder ohne Pflichtspalte."""


class ZeilenFehler(ValueError):
    """Eine Zeile ist unbrauchbar; grund ist einer aus GRUENDE oder "summenzeile"."""

    def __init__(self, grund: str, text: str) -> None:
        super().__init__(text)
        self.grund = grund


GRUENDE = ("pflichtwert_fehlt", "unlesbar", "unplausibel")


@dataclass(frozen=True)
class Format:
    """Wie ein Werk liefert: Dateimuster, Trennzeichen, Datum, Zahlen, Spalten und Schichten."""

    muster: str
    trenn: str
    datum: str
    dezimal: str
    fahrenheit: bool
    spalten: dict[str, str]
    schichten: dict[str, str]


FORMATE: dict[str, Format] = {
    "Illinois": Format(
        muster="IL_shift_report_*.csv",
        trenn=",",
        datum="%m/%d/%Y",
        dezimal=".",
        fahrenheit=True,
        spalten={
            "Date": "datum",
            "Line": "linie",
            "Shift": "schicht",
            "Planned Qty": "soll_stueck",
            "Actual Qty": "ist_stueck",
            "Scrap Qty": "ausschuss_stueck",
            "Headcount": "personal",
            "Hall Temp (F)": "hallentemp",
        },
        schichten={"1st": "Früh", "2nd": "Spät"},
    ),
    "Stammwerk": Format(
        muster="stammwerk_*.csv",
        trenn=";",
        datum="%d.%m.%Y",
        dezimal=",",
        fahrenheit=False,
        spalten={
            "Datum": "datum",
            "Linie": "linie",
            "Schicht": "schicht",
            "Soll": "soll_stueck",
            "Ist": "ist_stueck",
            "Ausschuss": "ausschuss_stueck",
            "Personal": "personal",
            "Hallentemperatur": "hallentemp",
        },
        schichten={"Früh": "Früh", "Spät": "Spät", "Nacht": "Nacht"},
    ),
}
SCHICHT_RANG = {"Früh": 0, "Spät": 1, "Nacht": 2}
PFLICHT = ("datum", "linie", "schicht", "soll_stueck", "ist_stueck", "ausschuss_stueck", "personal")

In [ ]:
for werk, fmt in FORMATE.items():
    einheit = "Fahrenheit" if fmt.fahrenheit else "Celsius"
    print(f"{werk}: {fmt.muster}, Trennzeichen {fmt.trenn!r}, Datum {fmt.datum}, Dezimalzeichen {fmt.dezimal!r}, {einheit}")
fehler = ZeilenFehler("unplausibel", "Ausschuss größer als Ist")
print(fehler.grund, "|", fehler, "|", isinstance(fehler, ValueError))

## A · Pfade und Kodierung
*Lerneinheit 4 · Skript Abschnitte 4.1 und 4.2*

Im Ordner `daten/roh` liegen 73 Dateien. Das Skript sucht dort `liesmich.txt` und die Dateigrößen, hier findest du die Lieferungen selbst und liest sie mit der richtigen Kodierung.

### TODO 1 · Lieferdateien finden und sortieren
**Ziel:** `lieferungen(eingang)` gibt alle Lieferdateien als Liste von Paaren `(werk, pfad)` zurück.

**Signatur:** `lieferungen(eingang: Path) -> list[tuple[str, Path]]`

**Regeln**
- Geh die Werke in der Reihenfolge `sorted(FORMATE)` durch, also erst `"Illinois"`, dann `"Stammwerk"`.
- Zu einem Werk gehören alle Dateien in `eingang`, deren Name zum Muster `FORMATE[werk].muster` passt (`Path.glob`).
- Sortiere die Dateien eines Werks nach Dateiname (`pfad.name`). Verlass dich nie auf die Reihenfolge, die das Betriebssystem liefert: Sie kann auf jedem Rechner anders sein (Skript Abschnitt 1.2).
- Dateien, die zu keinem Muster passen, gehören nicht dazu.

In [ ]:
# → src/nordwerk/importer.py
def lieferungen(eingang: Path) -> list[tuple[str, Path]]:
    """Alle Lieferdateien als (werk, pfad): Werke alphabetisch, je Werk nach Dateiname sortiert."""
    raise NotImplementedError("TODO 1")

In [ ]:
# Selbsttest TODO 1
zeige(lambda: f"{len(lieferungen(EINGANG))} Dateien, erste {lieferungen(EINGANG)[0][1].name}, letzte {lieferungen(EINGANG)[-1][1].name}")


def _t1_echt():
    liste = lieferungen(EINGANG)
    assert isinstance(liste, list) and all(isinstance(p, Path) for _, p in liste), "erwartet eine Liste von Paaren (werk, Path)"
    assert len(liste) == 72, f"{len(liste)} statt 72 Lieferdateien"
    assert "liesmich.txt" not in [p.name for _, p in liste], "liesmich.txt ist keine Lieferung"
    assert [w for w, _ in liste] == ["Illinois"] * 13 + ["Stammwerk"] * 59, "erst die 13 Dateien aus Illinois, dann die 59 aus dem Stammwerk"
    assert liste[0][1].name == "IL_shift_report_06292026.csv" and liste[-1][1].name == "stammwerk_2026-09-26.csv", "erste oder letzte Datei falsch"


class _Durcheinander(type(Path())):
    """Ein Ordner, der seine Dateien rückwärts aufzählt, wie es ein anderes Dateisystem dürfte."""

    def glob(self, muster, **k):
        return iter(sorted(super().glob(muster, **k), reverse=True))

    def iterdir(self):
        return iter(sorted(super().iterdir(), reverse=True))


def _t1_sortiert():
    namen = [p.name for _, p in lieferungen(_Durcheinander(EINGANG))]
    assert namen == sorted(namen[:13]) + sorted(namen[13:]), "Dateien kommen in der Reihenfolge des Betriebssystems, nicht nach Namen sortiert"


check("TODO 1 Lieferdateien", _t1_echt)
check("TODO 1 unabhängig vom Betriebssystem sortiert", _t1_sortiert)

Es sind 72 Lieferdateien, 13 aus Illinois und 59 aus dem Stammwerk. `liesmich.txt` passt zu keinem Muster; der Bericht in TODO 8 führt sie unter „ignoriert“. Der zweite Test gibt dir einen Ordner, der seine Dateien rückwärts aufzählt. Nur wer selbst sortiert, bekommt dort dieselbe Liste.

### TODO 2 · Text sicher lesen
Die Lieferungen nutzen drei Kodierungen: Illinois schreibt UTF-8 mit BOM, das Stammwerk bis zum 31. August 2026 Windows-1252 und nach dem Update des Fertigungssystems am 1. September UTF-8 ohne BOM. Im Skript hast du gesehen, was beim falschen Lesen passiert: ein `UnicodeDecodeError` oder `ï»¿` vor dem ersten Spaltennamen.

**Ziel:** `lies_text(pfad)` liest eine Lieferdatei als Text und meldet, welche Kodierung gepasst hat.

**Signatur:** `lies_text(pfad: Path) -> tuple[str, str]`, Rückgabe `(text, kodierung)`

**Regeln**
- Lies die Datei als Bytes (`pfad.read_bytes()`).
- Ist sie leer oder enthält sie nur Leerraum, löse `LieferungsFehler(f"{pfad.name}: Datei ist leer")` aus.
- Die Kodierung ist `"utf-8-sig"`, wenn die Datei mit der BOM (`codecs.BOM_UTF8`) beginnt, `"utf-8"`, wenn sie sich ohne BOM als UTF-8 lesen lässt, und sonst `"cp1252"`. Der Text enthält keine BOM.
- Überleg dir die Reihenfolge der Versuche (Skript Abschnitt 4.2): Windows-1252 liest jedes Byte ohne Fehler, auch die BOM.

In [ ]:
# → src/nordwerk/importer.py
def lies_text(pfad: Path) -> tuple[str, str]:
    """Liest eine Lieferdatei; liefert (text, kodierung) mit utf-8-sig, utf-8 oder cp1252."""
    raise NotImplementedError("TODO 2")

In [ ]:
# Selbsttest TODO 2
zeige(lambda: lies_text(EINGANG / "stammwerk_2026-06-29.csv")[0].splitlines()[1])


def _t2_alle():
    zaehlung = {}
    for pfad in sorted(EINGANG.glob("*.csv")):
        try:
            text, kodierung = lies_text(pfad)
        except LieferungsFehler:
            continue
        zaehlung[kodierung] = zaehlung.get(kodierung, 0) + 1
        if pfad.name.startswith("IL_"):
            kopf = text.splitlines()[0]
            assert kopf.startswith("Date,"), f"Illinois-Kopf beginnt mit {kopf[:7]!r}, BOM nicht entfernt"
        else:
            assert "Früh" in text, f"{pfad.name}: „Früh“ falsch gelesen ({text.splitlines()[1][:30]!r})"
    assert zaehlung == {"utf-8-sig": 13, "cp1252": 37, "utf-8": 21}, f"Kodierungen {zaehlung}, erwartet utf-8-sig 13, utf-8 21, cp1252 37"


def _t2_leer():
    try:
        lies_text(EINGANG / "stammwerk_2026-07-22.csv")
    except LieferungsFehler as e:
        assert str(e) == "stammwerk_2026-07-22.csv: Datei ist leer", f"Meldung {str(e)!r}"
    else:
        raise AssertionError("die leere Datei muss LieferungsFehler auslösen")


def _t2_klein():
    with tempfile.TemporaryDirectory() as ordner:
        faelle = {"bom.csv": codecs.BOM_UTF8 + "Spät".encode("utf-8"), "utf8.csv": "Spät".encode("utf-8"),
                  "ansi.csv": "Spät".encode("cp1252"), "leerraum.csv": b" \r\n\r\n"}
        for name, inhalt in faelle.items():
            (Path(ordner) / name).write_bytes(inhalt)
        for name, kodierung in [("bom.csv", "utf-8-sig"), ("utf8.csv", "utf-8"), ("ansi.csv", "cp1252")]:
            ergebnis = lies_text(Path(ordner) / name)
            assert ergebnis == ("Spät", kodierung), f"{name}: {ergebnis!r} statt ('Spät', {kodierung!r})"
        try:
            lies_text(Path(ordner) / "leerraum.csv")
        except LieferungsFehler:
            pass
        else:
            raise AssertionError("eine Datei nur mit Leerraum gilt als leer")


check("TODO 2 alle Lieferungen", _t2_alle)
check("TODO 2 leere Datei", _t2_leer)
check("TODO 2 kleine Beispiele", _t2_klein)

13 Dateien kommen in UTF-8 mit BOM, 21 in UTF-8 und 37 in Windows-1252. Die leere Datei vom 22. Juli zählt nicht mit, sie löst `LieferungsFehler` aus. In allen Stammwerk-Dateien steht jetzt „Früh“, egal ob das „ü“ als ein Byte oder als zwei kam.

## B · Datentyp und Zeilen
*Lerneinheit 5 · Skript Abschnitte 5.3, 5.4, 4.3 und 4.4*

Beide Werke melden dasselbe in verschiedener Form. Ab hier hat jede Schichtmeldung, egal woher, denselben Datentyp.

### TODO 3 · Eine Schichtmeldung als Datentyp
Das Skript baut `Lieferdatei` als `dataclass`. Hier baust du den Datentyp für eine Zeile der Ergebnistabelle.

**Ziel:** `Schicht` ist eine unveränderliche `dataclass` mit zehn Feldern und der Methode `schluessel()`.

**Regeln**
- `@dataclass(frozen=True)`, Felder genau in dieser Reihenfolge und mit diesen Typen: `werk: str`, `datum: dt.date`, `linie: str`, `schicht: str`, `soll_stueck: int`, `ist_stueck: int`, `ausschuss_stueck: int`, `personal: int`, `hallentemp_c: float | None`, `quelle: str`.
- `schluessel(self) -> tuple[str, dt.date, str, str]` gibt `(werk, datum, linie, schicht)` zurück. Daran erkennt TODO 8 Dubletten.
- `SPALTEN` unter der Klasse liest die Feldnamen mit `fields` aus. Die Reihenfolge der Felder ist damit die Reihenfolge der Spalten in `schichten.csv`.

In [ ]:
# → src/nordwerk/importer.py
@dataclass(frozen=True)
class Schicht:
    """Eine Schichtmeldung, in beiden Werken gleich aufgebaut."""

    # TODO 3: die zehn Felder mit Typannotation, in der Reihenfolge aus der Aufgabe

    def schluessel(self) -> tuple[str, dt.date, str, str]:
        """Der fachliche Schlüssel (werk, datum, linie, schicht)."""
        raise NotImplementedError("TODO 3")


SPALTEN = [f.name for f in fields(Schicht)]

In [ ]:
# Selbsttest TODO 3
print(SPALTEN)
ERWARTETE_FELDER = [("werk", str), ("datum", dt.date), ("linie", str), ("schicht", str), ("soll_stueck", int), ("ist_stueck", int),
                    ("ausschuss_stueck", int), ("personal", int), ("hallentemp_c", float | None), ("quelle", str)]


def _typ_ok(ist, soll):
    if isinstance(ist, str):           # bei from __future__ import annotations stehen Typen als Text da
        schreibweisen = {str: {"str"}, int: {"int"}, dt.date: {"dt.date", "date", "datetime.date"},
                         float | None: {"float|None", "None|float", "Optional[float]", "typing.Optional[float]"}}
        return ist.replace(" ", "") in schreibweisen[soll]
    return ist == soll


def _t3_felder():
    assert is_dataclass(Schicht), "Schicht ist keine dataclass"
    if not fields(Schicht):
        raise NotImplementedError("TODO 3")
    ist = [(f.name, f.type) for f in fields(Schicht)]
    assert [n for n, _ in ist] == [n for n, _ in ERWARTETE_FELDER], f"Felder {[n for n, _ in ist]}, erwartet {[n for n, _ in ERWARTETE_FELDER]}"
    falsch = [n for (n, t), (_, soll) in zip(ist, ERWARTETE_FELDER) if not _typ_ok(t, soll)]
    assert not falsch, f"Typannotation stimmt nicht bei {falsch}"
    assert SPALTEN == [n for n, _ in ERWARTETE_FELDER], "SPALTEN neu ausführen: es muss die Feldnamen von Schicht enthalten"


def _t3_verhalten():
    if not fields(Schicht):
        raise NotImplementedError("TODO 3")
    s = Schicht("Stammwerk", dt.date(2026, 7, 14), "3", "Spät", 249, 218, 4, 7, None, "stammwerk_2026-07-14.csv")
    assert s.schluessel() == ("Stammwerk", dt.date(2026, 7, 14), "3", "Spät"), f"schluessel() liefert {s.schluessel()!r}"
    try:
        s.ist_stueck = 0
    except FrozenInstanceError:
        pass
    else:
        raise AssertionError("Schicht lässt sich ändern: frozen=True fehlt")
    assert s == Schicht("Stammwerk", dt.date(2026, 7, 14), "3", "Spät", 249, 218, 4, 7, None, "stammwerk_2026-07-14.csv"), "gleiche Werte sollen gleich sein"


check("TODO 3 Felder und Typen", _t3_felder)
check("TODO 3 schluessel und frozen", _t3_verhalten)

`SPALTEN` hat jetzt zehn Namen. Weil die Klasse `frozen` ist, kann kein späterer Schritt eine Meldung aus Versehen ändern, und `schluessel()` sagt an einer Stelle, was „dieselbe Schicht“ heißt. Die Typannotationen prüft Python beim Ausführen nicht, `Schicht("Illinois", "gestern", …)` geht durch. Das finden mypy und Pylance (Skript Abschnitt 5.4).

### TODO 4 · Datum und Temperatur je Werk
Das Skript liest das Datum aus dem Dateinamen. Hier liest du es aus den Zellen der Lieferung, mit dem Format des Werks.

**Ziel:** zwei kleine Funktionen, die einen Zellinhalt nach dem Format `fmt` eines Werks lesen.

**Signaturen:** `lies_datum(text: str, fmt: Format) -> dt.date` und `lies_temperatur(text: str, fmt: Format) -> float | None`

**Regeln**
- `lies_datum`: Leerraum am Rand entfernen und mit `dt.datetime.strptime(…, fmt.datum).date()` lesen. Ein falsches Datum löst `ValueError` aus, das ist richtig so.
- `lies_temperatur`: Leerraum entfernen; leer heißt „nicht gemessen“ und ergibt `None` (nicht 0). Sonst das Dezimalzeichen `fmt.dezimal` beachten, als `float` lesen, bei `fmt.fahrenheit` in Grad Celsius umrechnen, $C = (F - 32) \cdot 5/9$, und zum Schluss auf eine Nachkommastelle runden: `round(…, 1)`.

In [ ]:
# → src/nordwerk/importer.py
def lies_datum(text: str, fmt: Format) -> dt.date:
    """Liest ein Datum im Format des Werks."""
    raise NotImplementedError("TODO 4")


def lies_temperatur(text: str, fmt: Format) -> float | None:
    """Liest eine Hallentemperatur in Grad Celsius mit einer Nachkommastelle; leer ergibt None."""
    raise NotImplementedError("TODO 4")

In [ ]:
# Selbsttest TODO 4
IL, SW = FORMATE["Illinois"], FORMATE["Stammwerk"]
zeige(lambda: (lies_datum("07/02/2026", IL), lies_temperatur("85.7", IL), lies_temperatur("23,4", SW)))


def _t4_datum():
    assert lies_datum("07/02/2026", IL) == dt.date(2026, 7, 2), f"07/02/2026 in Illinois ist der 2. Juli, nicht {lies_datum('07/02/2026', IL)}"
    assert lies_datum(" 14.07.2026 ", SW) == dt.date(2026, 7, 14), "Stammwerk-Datum mit Leerraum nicht gelesen"
    for text, fmt in [("31.06.2026", SW), ("2026-07-02", IL)]:
        try:
            lies_datum(text, fmt)
        except ValueError:
            pass
        else:
            raise AssertionError(f"{text!r} ist kein gültiges Datum und muss ValueError auslösen")


def _t4_temperatur():
    faelle = [("85.7", IL, 29.8), ("85.1", IL, 29.5), ("32.0", IL, 0.0), ("23,4", SW, 23.4), (" 19,1 ", SW, 19.1), ("22", SW, 22.0)]
    for text, fmt, soll in faelle:
        ist = lies_temperatur(text, fmt)
        assert isinstance(ist, float), f"{text!r} ergibt {ist!r}, erwartet eine Zahl"
        if fmt is IL and abs(ist - (float(text) - 32) * 5 / 9) < 1e-9 and ist != soll:
            raise AssertionError(f"{text!r} °F ergibt {ist!r}: umgerechnet, aber nicht auf eine Stelle gerundet")
        assert ist == soll, f"{text!r} ergibt {ist!r}, erwartet {soll}" + (" (Fahrenheit nicht umgerechnet?)" if fmt is IL else "")
    for leer in ["", "  "]:
        assert lies_temperatur(leer, SW) is None, f"leere Temperatur ergibt {lies_temperatur(leer, SW)!r}, erwartet None (nicht gemessen ist nicht 0 Grad)"


check("TODO 4 Datum je Werk", _t4_datum)
check("TODO 4 Temperatur je Werk", _t4_temperatur)

`07/02/2026` ist in Illinois der 2. Juli, im Stammwerk wäre `07.02.2026` der 7. Februar: Ohne das Format des Werks wäre das Datum still falsch. 85,7 °F sind 29,8 °C. Ohne Umrechnung sähe Illinois um rund 60 Grad wärmer aus als das Stammwerk. Gerundet wird erst am Ende, damit sich keine Rundungsfehler aufaddieren und `schichten.csv` überall dieselben Ziffern zeigt.

### TODO 5 · Eine Zeile prüfen
Das Skript zeigt `try`, `except`, `else`, `finally` und `raise … from e` an der Konfiguration. Hier entscheidest du für eine Zeile der Lieferung, ob sie in die Tabelle darf.

**Ziel:** `zeile(werte, werk, fmt, quelle)` macht aus den Zellen einer Zeile eine `Schicht` oder löst `ZeilenFehler` mit dem passenden `grund` aus.

**Signatur:** `zeile(werte: dict[str, str], werk: str, fmt: Format, quelle: str) -> Schicht`. `werte` hat die Feldnamen als Schlüssel (`"datum"`, `"linie"`, …, `"hallentemp"`), die Werte sind Text wie in der Datei.

**Regeln, in dieser Reihenfolge**
1. Steht in `datum` (ohne Leerraum) `TOTAL`, ist es die Summenzeile von Illinois: `ZeilenFehler("summenzeile", …)`.
2. Ist ein Feld aus `PFLICHT` leer (oder fehlt es), `ZeilenFehler("pflichtwert_fehlt", …)`. Die Temperatur ist keine Pflicht.
3. Baue die `Schicht`: Datum mit `lies_datum`, Linie ohne Leerraum, Schicht über `fmt.schichten` (`"1st"` → `"Früh"`), die vier Zahlen mit `int`, Temperatur mit `lies_temperatur`. Löst dabei etwas `ValueError` oder `KeyError` aus, wird daraus `ZeilenFehler("unlesbar", …)`, und zwar mit `raise … from e`, damit die ursprüngliche Ausnahme als `__cause__` erhalten bleibt.
4. Ist eine Zahl negativ oder der Ausschuss größer als die Ist-Stückzahl, `ZeilenFehler("unplausibel", …)`.

Der Text der Ausnahme landet später im Log; schreib ihn so, dass man den Fehler versteht, etwa „Ausschuss 961 größer als Ist 169“.

In [ ]:
# → src/nordwerk/importer.py
def zeile(werte: dict[str, str], werk: str, fmt: Format, quelle: str) -> Schicht:
    """Prüft eine Zeile und liefert die Schicht; sonst ZeilenFehler mit grund."""
    raise NotImplementedError("TODO 5")

In [ ]:
# Selbsttest TODO 5
GUT_IL = {"datum": "06/29/2026", "linie": "A", "schicht": "1st", "soll_stueck": "178", "ist_stueck": "165",
          "ausschuss_stueck": "7", "personal": "12", "hallentemp": "85.1"}
GUT_SW = {"datum": "14.07.2026", "linie": " 3 ", "schicht": "Nacht", "soll_stueck": "203", "ist_stueck": "164",
          "ausschuss_stueck": "2", "personal": "6", "hallentemp": ""}
zeige(lambda: zeile(GUT_IL, "Illinois", IL, "IL_shift_report_06292026.csv"))


def _grund(werte, werk="Stammwerk"):
    try:
        zeile(werte, werk, FORMATE[werk], "probe.csv")
    except ZeilenFehler as e:
        assert isinstance(e, ValueError), "ZeilenFehler muss von ValueError erben"
        return e
    return None


def _t5_gut():
    braucht("TODO 3", "TODO 4")
    s = zeile(GUT_IL, "Illinois", IL, "IL_shift_report_06292026.csv")
    assert s == Schicht("Illinois", dt.date(2026, 6, 29), "A", "Früh", 178, 165, 7, 12, 29.5, "IL_shift_report_06292026.csv"), f"Illinois-Zeile ergibt {s}"
    s = zeile(GUT_SW, "Stammwerk", SW, "stammwerk_2026-07-14.csv")
    assert s.linie == "3" and s.schicht == "Nacht" and s.hallentemp_c is None, f"Stammwerk-Zeile ergibt {s}"


def _t5_fehler():
    braucht("TODO 3", "TODO 4")
    faelle = [
        ("Summenzeile", {**dict.fromkeys(GUT_IL, ""), "datum": "TOTAL", "soll_stueck": "2314", "ist_stueck": "2016", "ausschuss_stueck": "61"}, "Illinois", "summenzeile"),
        ("leere Ist-Stückzahl", {**GUT_IL, "ist_stueck": ""}, "Illinois", "pflichtwert_fehlt"),
        ("Personal nur Leerraum", {**GUT_SW, "personal": "  "}, "Stammwerk", "pflichtwert_fehlt"),
        ("Datum 31.06.", {**GUT_SW, "datum": "31.06.2026"}, "Stammwerk", "unlesbar"),
        ("Schicht 3rd", {**GUT_IL, "schicht": "3rd"}, "Illinois", "unlesbar"),
        ("Stückzahl 16a", {**GUT_SW, "ist_stueck": "16a"}, "Stammwerk", "unlesbar"),
        ("Ausschuss 961 bei Ist 169", {**GUT_SW, "ist_stueck": "169", "ausschuss_stueck": "961"}, "Stammwerk", "unplausibel"),
        ("negatives Soll", {**GUT_SW, "soll_stueck": "-5"}, "Stammwerk", "unplausibel"),
    ]
    for name, werte, werk, soll in faelle:
        e = _grund(werte, werk)
        assert e is not None, f"{name}: keine Ausnahme, erwartet ZeilenFehler mit grund {soll!r}"
        assert getattr(e, "grund", None) == soll, f"{name}: grund {getattr(e, 'grund', None)!r}, erwartet {soll!r}"
        if soll == "unlesbar":
            assert isinstance(e.__cause__, (ValueError, KeyError)), f"{name}: __cause__ ist {e.__cause__!r}, raise … from e fehlt"


check("TODO 5 gültige Zeilen", _t5_gut)
check("TODO 5 Fehlerarten mit grund", _t5_fehler)

Eine Zeile kann auf vier Arten scheitern, und jede hat ihren eigenen `grund`. Die Summenzeile ist kein Fehler der Lieferung, sie wird in TODO 6 nur gezählt. Wegen `raise … from e` zeigt ein Traceback später beides: was an der Zeile falsch war und welcher Aufruf es gemerkt hat.

## C · Generator und Logging
*Lerneinheiten 4 und 5 · Skript Abschnitte 5.1 und 4.5*

### TODO 6 · Zeilen als Generator
Das Skript schreibt einen Generator über die Zeilen von `liesmich.txt`. Hier liefert ein Generator die geprüften Schichten einer Lieferdatei, eine nach der anderen.

**Ziel:** `zeilen(pfad, werk, zaehler)` ist eine Generatorfunktion: Sie gibt mit `yield` jede gültige `Schicht` der Datei zurück und zählt in `zaehler` mit, was sonst passiert.

**Signatur:** `zeilen(pfad: Path, werk: str, zaehler: dict[str, int]) -> Iterator[Schicht]`

**Regeln**
- Text mit `lies_text` lesen und die Kodierung in `zaehler` zählen (`zaehler[kodierung]` um 1 erhöhen, fehlende Schlüssel beginnen bei 0).
- `csv.reader(text.splitlines(), delimiter=fmt.trenn)`; die erste Zeile ist der Kopf, jeden Spaltennamen ohne Leerraum am Rand.
- Fehlt eine Spalte aus `fmt.spalten` im Kopf, löse `LieferungsFehler(f"{pfad.name}: Pflichtspalte fehlt: {…}")` aus, mehrere Spalten durch Komma und Leerzeichen getrennt.
- Leere Zeilen überspringen. Für jede andere Zeile ein Wörterbuch `werte` mit den Feldnamen bauen (fehlt hinten eine Zelle, ist der Wert `""`) und `yield zeile(werte, werk, fmt, pfad.name)`.
- `ZeilenFehler` abfangen, `zaehler[e.grund]` um 1 erhöhen und melden: `log.warning("%s Zeile %d: %s", pfad.name, nr, e)`, bei der Summenzeile nur `log.debug(…)`. `nr` ist die Zeilennummer in der Datei, der Kopf ist Zeile 1.

In [ ]:
# → src/nordwerk/importer.py
def zeilen(pfad: Path, werk: str, zaehler: dict[str, int]) -> Iterator[Schicht]:
    """Liefert die gültigen Schichten einer Lieferdatei; zählt Kodierung und Gründe in zaehler."""
    raise NotImplementedError("TODO 6")

In [ ]:
# Selbsttest TODO 6
zeige(lambda: len(list(zeilen(EINGANG / "IL_shift_report_06292026.csv", "Illinois", {}))))


def _t6_illinois():
    braucht("TODO 2", "TODO 3", "TODO 4", "TODO 5")
    assert inspect.isgeneratorfunction(zeilen), "zeilen ist keine Generatorfunktion (yield fehlt)"
    zaehler = {}
    liste, meldungen = mitschreiben(lambda: list(zeilen(EINGANG / "IL_shift_report_06292026.csv", "Illinois", zaehler)))
    assert len(liste) == 16, f"{len(liste)} Schichten statt 16 (Montag bis Donnerstag, je zwei Linien und zwei Schichten)"
    assert liste[0] == Schicht("Illinois", dt.date(2026, 6, 29), "A", "Früh", 178, 165, 7, 12, 29.5, "IL_shift_report_06292026.csv"), f"erste Schicht {liste[0]}"
    assert zaehler == {"utf-8-sig": 1, "summenzeile": 1}, f"zaehler {zaehler}, erwartet {{'utf-8-sig': 1, 'summenzeile': 1}}"
    assert [m[0] for m in meldungen] == ["DEBUG"], f"Meldungen {meldungen}: die Summenzeile gehört nur ins DEBUG-Log"
    assert meldungen[0][2].startswith("IL_shift_report_06292026.csv Zeile 18: "), f"Meldung {meldungen[0][2]!r}: Zeile 18 erwartet (Kopf ist Zeile 1)"


def _t6_warnung():
    braucht("TODO 2", "TODO 3", "TODO 4", "TODO 5")
    zaehler = {}
    liste, meldungen = mitschreiben(lambda: list(zeilen(EINGANG / "stammwerk_2026-09-16.csv", "Stammwerk", zaehler)))
    assert len(liste) == 11 and zaehler == {"utf-8": 1, "unplausibel": 1}, f"{len(liste)} Schichten, zaehler {zaehler}"
    warnungen = [m for m in meldungen if m[0] == "WARNING"]
    assert len(warnungen) == 1 and warnungen[0][1] == "nordwerk.importer" and warnungen[0][2].startswith("stammwerk_2026-09-16.csv Zeile 11: "), f"Warnungen {warnungen}"


def _t6_kopf():
    braucht("TODO 2", "TODO 3", "TODO 4", "TODO 5")
    gen = zeilen(EINGANG / "IL_shift_report_08102026.csv", "Illinois", {})
    try:
        next(gen)
    except LieferungsFehler as e:
        assert str(e) == "IL_shift_report_08102026.csv: Pflichtspalte fehlt: Scrap Qty", f"Meldung {str(e)!r}"
    else:
        raise AssertionError("die Datei ohne 'Scrap Qty' muss beim ersten next() LieferungsFehler auslösen")
    with tempfile.TemporaryDirectory() as ordner:
        pfad = Path(ordner) / "stammwerk_probe.csv"
        pfad.write_bytes("Datum ; Linie;Schicht;Soll;Ist;Ausschuss;Personal;Hallentemperatur\r\n\r\n01.09.2026;1;Früh;10;9;1;5\r\n".encode("utf-8"))
        zaehler = {}
        liste = list(zeilen(pfad, "Stammwerk", zaehler))
        assert len(liste) == 1 and liste[0].hallentemp_c is None and zaehler == {"utf-8": 1}, f"kleines Beispiel mit Leerraum im Kopf, Leerzeile und fehlender letzter Zelle: {liste}, {zaehler}"


check("TODO 6 Generator über eine Illinois-Datei", _t6_illinois)
check("TODO 6 Warnung mit Zeilennummer", _t6_warnung)
check("TODO 6 Kopf prüfen und kleine Beispiele", _t6_kopf)

Die erste Illinois-Datei liefert 16 Schichten: vier Tage, zwei Linien, zwei Schichten. Am Freitag, dem 3. Juli 2026, ruhte die Arbeit (Independence Day, nachgeholt). Die Summenzeile `TOTAL` steht in Zeile 18 und erscheint nur im DEBUG-Log. Die Datei ohne `Scrap Qty` meldet sich erst beim ersten `next()`: Ein Generator läuft erst los, wenn jemand den ersten Wert will.

### Das Notebook als Paket
Ab TODO 7 entsteht `cli.py`. Im Paket holt `cli.py` die anderen Module mit `from nordwerk import __version__, importer, konfig` und ruft dann etwa `importer.importiere(...)` auf. Im Notebook gibt es dieses Paket nicht, alle Zellen teilen sich einen Namensraum. Die nächste Zelle spielt deshalb das Paket: Sie meldet `nordwerk`, `nordwerk.importer` und `nordwerk.konfig` als Module an, die jeden Namen beim Zugriff aus dem Notebook holen. Diese Zelle trägt keine Markierung und kommt nicht ins Repository; im Repository gibt es das echte Paket. Ist `nordwerk` in deiner `.venv` schon installiert, verdeckt die Zelle es für diese Sitzung des Kernels: Das Notebook prüft immer seinen eigenen Code.

In [ ]:
# Nur im Notebook: das Paket nordwerk spielen. Nicht ins Repository übernehmen.
import sys
import types


def _aus_dem_notebook(name):
    try:
        return globals()[name]
    except KeyError:
        raise AttributeError(name) from None


_paket = types.ModuleType("nordwerk")
_paket.__path__ = []
_paket.__version__ = "0.1.0"
for _teil in ("importer", "konfig"):
    _modul = types.ModuleType(f"nordwerk.{_teil}")
    _modul.__getattr__ = _aus_dem_notebook
    setattr(_paket, _teil, _modul)
    sys.modules[_modul.__name__] = _modul
sys.modules["nordwerk"] = _paket
print("Paket nordwerk im Notebook bereit:", sorted(name for name in sys.modules if name.startswith("nordwerk")))

### TODO 7 · Logging einrichten
Bibliothekscode meldet über `log`, wer das Programm startet, entscheidet, was davon erscheint. Das Skript zeigt Stufen, Handler und die Falle des doppelt angehängten Handlers. Hier schreibst du die Einrichtung für das Paket. Sie ist die erste Zelle von `cli.py` und enthält deshalb alle Importe dieses Moduls.

**Ziel:** `richte_logging_ein(stufe, ziel)` richtet den Logger `"nordwerk"` so ein, dass jede Meldung genau einmal erscheint, auch wenn du die Funktion mehrmals aufrufst.

**Signatur:** `richte_logging_ein(stufe: str = "WARNING", ziel: TextIO | None = None) -> logging.Logger`

**Regeln**
- Logger `logging.getLogger("nordwerk")`; er gilt auch für `nordwerk.importer`, denn Logger bilden über die Punkte im Namen einen Baum.
- Stufe setzen, Groß- und Kleinschreibung egal (`stufe.upper()`). Eine unbekannte Stufe löst dabei `ValueError` aus, das ist richtig so.
- **Vorhandene Handler entfernen**, dann genau einen `logging.StreamHandler` anhängen. Er schreibt nach `ziel`, ohne `ziel` nach `sys.stderr`.
- Format des Handlers: `"%(levelname)s %(name)s: %(message)s"`.
- `propagate = False`: Die Meldungen gehen nicht zusätzlich an den Wurzel-Logger, sonst erscheinen sie in manchen Umgebungen doppelt.
- Den Logger zurückgeben.

In [ ]:
# → src/nordwerk/cli.py
"""Kommandozeile des Pakets nordwerk: nordwerk import."""

import argparse
import json
import logging
import sys
import tomllib
from pathlib import Path
from typing import TextIO

from nordwerk import __version__, importer, konfig


def richte_logging_ein(stufe: str = "WARNING", ziel: TextIO | None = None) -> logging.Logger:
    """Richtet den Logger "nordwerk" mit genau einem Handler ein; ohne ziel nach sys.stderr."""
    raise NotImplementedError("TODO 7")

In [ ]:
# Selbsttest TODO 7
def _t7_doppelt():
    with logger_gesichert():
        puffer = io.StringIO()
        richte_logging_ein("INFO", puffer)
        logger = richte_logging_ein("INFO", puffer)
        assert logger is logging.getLogger("nordwerk"), "richte_logging_ein soll den Logger 'nordwerk' zurückgeben"
        logging.getLogger("nordwerk.importer").info("Probe %d", 1)
        ausgabe = puffer.getvalue().splitlines()
        assert len(logger.handlers) == 1, f"{len(logger.handlers)} Handler nach zwei Aufrufen: vorhandene Handler nicht entfernt"
    assert ausgabe == ["INFO nordwerk.importer: Probe 1"], f"Ausgabe {ausgabe}, erwartet genau eine Zeile 'INFO nordwerk.importer: Probe 1'"


def _t7_stufe():
    wurzel = logging.getLogger()
    sammler = _Sammler()
    wurzel.addHandler(sammler)
    try:
        with logger_gesichert():
            puffer = io.StringIO()
            richte_logging_ein("warning", puffer)
            logging.getLogger("nordwerk.importer").info("unsichtbar")
            logging.getLogger("nordwerk.importer").warning("sichtbar")
    finally:
        wurzel.removeHandler(sammler)
    assert puffer.getvalue().splitlines() == ["WARNING nordwerk.importer: sichtbar"], f"mit WARNING: {puffer.getvalue().splitlines()}"
    assert not sammler.meldungen, "die Meldung kam zusätzlich beim Wurzel-Logger an: propagate = False fehlt"


def _t7_lauf():
    braucht("TODO 1", "TODO 6")
    with logger_gesichert():
        puffer = io.StringIO()
        richte_logging_ein("WARNING", puffer)
        for werk, pfad in lieferungen(EINGANG):
            try:
                for _ in zeilen(pfad, werk, {}):
                    pass
            except LieferungsFehler as e:
                logging.getLogger("nordwerk.importer").error("%s", e)
    stufen = [z.split(" ", 1)[0] for z in puffer.getvalue().splitlines()]
    assert stufen.count("WARNING") == 4 and stufen.count("ERROR") == 2 and len(stufen) == 6, f"Importlauf mit WARNING: {stufen}"


check("TODO 7 zweiter Aufruf ohne doppelte Zeilen", _t7_doppelt)
check("TODO 7 Stufe und propagate", _t7_stufe)
check("TODO 7 ein Importlauf mit WARNING", _t7_lauf)

Ab hier zeigt jede Zelle, die etwas meldet, die Meldungen genau einmal. Die nächste Zelle richtet das Logging für den Rest des Notebooks mit der Stufe `WARNING` ein. Ein Lauf über alle Lieferungen meldet so vier Warnungen (drei fehlende Ist-Stückzahlen, ein Ausschuss größer als Ist) und zwei Fehler (die leere Datei und die fehlende Spalte). Die Summenzeilen stehen nur im DEBUG-Log, die Zusammenfassung je Datei im INFO-Log.

In [ ]:
zeige(lambda: f"Logger {richte_logging_ein('WARNING').name}: Stufe WARNING, {len(logging.getLogger('nordwerk').handlers)} Handler")

## D · Zusammenführen und Prüfsumme
*Lerneinheiten 1 und 4 · Skript Abschnitte 1.3, 4.4 und 7.5*

### TODO 8 · Alles zusammenführen
Das Skript zeigt den Weg der Daten als Abbildung: 72 Dateien, 70 gelesen, 908 Zeilen, 892 übernommen. Hier schreibst du die Funktion, die ihn geht.

**Ziel:** `importiere(eingang)` liest alle Lieferungen, führt beide Werke in einer sortierten Liste zusammen und erstellt den Qualitätsbericht.

**Signatur:** `importiere(eingang: Path) -> tuple[list[Schicht], dict[str, Any]]`

**Regeln**
- Alle Dateien aus `lieferungen(eingang)` mit `zeilen` lesen, ein gemeinsames Wörterbuch `zaehler` für alle Dateien.
- Löst eine Datei `LieferungsFehler` aus, meldest du ihn mit `log.error("%s", e)`, nimmst die Datei mit `{"datei": name, "grund": …}` in die Liste `abgelehnt` auf (der Grund ohne den Dateinamen davor, etwa `"Datei ist leer"`) und machst mit der nächsten Datei weiter. Zeilen einer abgelehnten Datei zählen nicht.
- Für jede gelesene Datei eine INFO-Meldung, etwa `log.info("%s: %d Schichten", name, anzahl)`.
- **Dubletten** erkennst du an `schluessel()`: Die zuerst gelesene Schicht bleibt, jede weitere mit demselben Schlüssel zählt als Dublette.
- Ergebnis sortiert nach `(werk, datum, linie, SCHICHT_RANG[schicht])`. Alphabetisch käme „Nacht“ vor „Spät“.
- Der Bericht hat genau diese Schlüssel in dieser Reihenfolge (die Prüfsumme kommt in TODO 12 dazu):
```
{"dateien": {"gelesen": …, "abgelehnt": [{"datei": …, "grund": …}, …], "ignoriert": [Dateinamen]},
 "kodierung": {"utf-8-sig": …, "utf-8": …, "cp1252": …},
 "zeilen": {"gelesen": …, "verworfen": {"pflichtwert_fehlt": …, "unlesbar": …, "unplausibel": …},
            "dubletten": …, "uebernommen": …, "summenzeilen": …},
 "werke": {"Illinois": {"zeilen": …, "von": "JJJJ-MM-TT", "bis": "JJJJ-MM-TT", "ohne_temperatur": …},
           "Stammwerk": {…}}}
```
- `dateien.gelesen` zählt die Dateien ohne Ablehnung, `ignoriert` die Dateien im Eingangsordner, die zu keinem Muster passen, nach Namen sortiert. `zeilen.gelesen` zählt die Datenzeilen ohne Summenzeilen: übernommen plus verworfen plus Dubletten. `werke` gilt für die übernommenen Schichten; `von` und `bis` sind das erste und letzte Datum als Text, `ohne_temperatur` die Schichten ohne Hallentemperatur.

In [ ]:
# → src/nordwerk/importer.py
def importiere(eingang: Path) -> tuple[list[Schicht], dict[str, Any]]:
    """Liest alle Lieferungen, entfernt Dubletten, sortiert und liefert (schichten, bericht)."""
    raise NotImplementedError("TODO 8")

In [ ]:
# Selbsttest TODO 8
zeige(lambda: "\n".join(f"{k}: {json.dumps(v, ensure_ascii=False)}" for k, v in importiere(EINGANG)[1].items()))
BERICHT_SOLL = {
    "dateien": {"gelesen": 70, "abgelehnt": [{"datei": "IL_shift_report_08102026.csv", "grund": "Pflichtspalte fehlt: Scrap Qty"},
                                             {"datei": "stammwerk_2026-07-22.csv", "grund": "Datei ist leer"}], "ignoriert": ["liesmich.txt"]},
    "kodierung": {"utf-8-sig": 13, "utf-8": 21, "cp1252": 37},
    "zeilen": {"gelesen": 908, "verworfen": {"pflichtwert_fehlt": 3, "unlesbar": 0, "unplausibel": 1}, "dubletten": 12,
               "uebernommen": 892, "summenzeilen": 12},
    "werke": {"Illinois": {"zeilen": 229, "von": "2026-06-29", "bis": "2026-09-25", "ohne_temperatur": 0},
              "Stammwerk": {"zeilen": 663, "von": "2026-06-29", "bis": "2026-09-26", "ohne_temperatur": 2}},
}


def _vergleiche(ist, soll, wo="bericht"):
    assert isinstance(ist, dict), f"{wo} ist {type(ist).__name__}, erwartet ein Wörterbuch"
    assert list(ist) == list(soll), f"{wo}: Schlüssel {list(ist)}, erwartet {list(soll)} in dieser Reihenfolge"
    for k, v in soll.items():
        if isinstance(v, dict):
            _vergleiche(ist[k], v, f"{wo}[{k!r}]")
        else:
            assert ist[k] == v, f"{wo}[{k!r}] ist {ist[k]!r}, erwartet {v!r}"


def _t8_echt():
    braucht("TODO 1", "TODO 2", "TODO 3", "TODO 4", "TODO 5", "TODO 6")
    (schichten, bericht), meldungen = mitschreiben(lambda: importiere(EINGANG))
    _vergleiche(bericht, BERICHT_SOLL)
    z = bericht["zeilen"]
    assert z["gelesen"] - sum(z["verworfen"].values()) - z["dubletten"] == z["uebernommen"] == len(schichten), "Bilanz stimmt nicht"
    assert schichten == sorted(schichten, key=lambda s: (s.werk, s.datum, s.linie, SCHICHT_RANG[s.schicht])), "Schichten nicht nach Werk, Datum, Linie und Schicht sortiert"
    erste = [s for s in schichten if s.werk == "Stammwerk" and s.datum == dt.date(2026, 8, 5)][0]
    assert erste.quelle == "stammwerk_2026-08-05.csv", f"bei Dubletten bleibt die zuerst gelesene Zeile, nicht die aus {erste.quelle}"
    stufen = [m[0] for m in meldungen]
    assert stufen.count("WARNING") == 4 and stufen.count("ERROR") == 2 and stufen.count("INFO") == 70, \
        f"Meldungen: {stufen.count('WARNING')} Warnungen, {stufen.count('ERROR')} Fehler, {stufen.count('INFO')} INFO (erwartet 4, 2, 70)"


def _t8_klein():
    braucht("TODO 1", "TODO 2", "TODO 3", "TODO 4", "TODO 5", "TODO 6")
    kopf = "Datum;Linie;Schicht;Soll;Ist;Ausschuss;Personal;Auslastung;Hallentemperatur\n"
    with tempfile.TemporaryDirectory() as ordner:
        o = Path(ordner)
        (o / "stammwerk_2026-09-02.csv").write_text(kopf + "02.09.2026;1;Spät;10;9;0;5;0,9;21,0\n02.09.2026;1;Nacht;10;9;0;5;0,9;20,0\n"
                                                    "01.09.2026;1;Früh;10;8;0;5;0,9;22,0\n", encoding="utf-8")
        (o / "stammwerk_2026-09-01.csv").write_text(kopf + "01.09.2026;1;Früh;10;9;1;5;0,9;23,0\n", encoding="utf-8")
        (o / "stammwerk_2026-09-03.csv").write_bytes(b"")
        (o / "notizen.txt").write_text("nur Notizen", encoding="utf-8")
        (schichten, bericht), _ = mitschreiben(lambda: importiere(o))
    folge = [(s.datum.day, s.schicht) for s in schichten]
    assert folge == [(1, "Früh"), (2, "Spät"), (2, "Nacht")], f"Reihenfolge {folge}, erwartet Früh, Spät, Nacht nach Datum"
    assert schichten[0].ist_stueck == 9 and schichten[0].quelle == "stammwerk_2026-09-01.csv", "bei Dubletten bleibt die zuerst gelesene Zeile"
    assert bericht["zeilen"]["dubletten"] == 1 and bericht["zeilen"]["gelesen"] == 4, f"zeilen {bericht['zeilen']}"
    assert bericht["dateien"] == {"gelesen": 2, "abgelehnt": [{"datei": "stammwerk_2026-09-03.csv", "grund": "Datei ist leer"}], "ignoriert": ["notizen.txt"]}, f"dateien {bericht['dateien']}"
    assert bericht["werke"]["Illinois"]["zeilen"] == 0, "Illinois hat hier keine Zeilen"


check("TODO 8 alle Lieferungen", _t8_echt)
check("TODO 8 kleines Beispiel mit Dublette und leerer Datei", _t8_klein)

70 Dateien sind gelesen, 2 abgelehnt. Von 908 Datenzeilen fallen 4 weg (3 ohne Ist-Stückzahl, 1 unplausibel), 12 sind Dubletten: `stammwerk_2026-08-06.csv` enthält zusätzlich die zwölf Zeilen vom 5. August. Es bleiben 892 Schichten, 229 aus Illinois und 663 aus dem Stammwerk. Die Bilanz geht auf: 908 − 4 − 12 = 892. Zwei Stammwerk-Schichten vom 14. Juli (Linie 3, Spät und Nacht) haben keine Hallentemperatur. Sie bleiben in der Tabelle, das Feld bleibt leer. Für sich genommen ist keine dieser Zeilen falsch; erst der Bericht zeigt, was die Lieferungen wirklich enthalten.

### TODO 9 · Tabelle schreiben und Prüfsumme
Das Skript zeigt an kleinen Bytes, wie Zeilenende und Kodierung die Prüfsumme ändern. Hier legst du `schichten.csv` Byte für Byte fest.

**Ziel:** `schreibe_csv(schichten)` liefert den Inhalt von `schichten.csv` als Bytes, `pruefsumme(daten)` die ersten zwölf Zeichen des SHA-256-Hashes.

**Signaturen:** `schreibe_csv(schichten: list[Schicht]) -> bytes` und `pruefsumme(daten: bytes) -> str`

**Regeln**
- Kopfzeile `SPALTEN`, Trennzeichen Komma, Zeilenende `\n` (Vorsicht: `csv.writer` schreibt ohne Angabe `\r\n`, also `lineterminator="\n"`).
- Datum als `JJJJ-MM-TT` (`isoformat()`), Temperatur mit genau einer Nachkommastelle (`f"{x:.1f}"`) oder leer, alle übrigen Werte wie sie sind.
- Schreib in einen `io.StringIO`-Puffer und kodiere am Ende als UTF-8 ohne BOM.
- `pruefsumme`: `hashlib.sha256(daten).hexdigest()[:12]`.

In [ ]:
# → src/nordwerk/importer.py
def schreibe_csv(schichten: list[Schicht]) -> bytes:
    """Inhalt von schichten.csv: Kopf SPALTEN, Komma, Zeilenende LF, UTF-8 ohne BOM."""
    raise NotImplementedError("TODO 9")


def pruefsumme(daten: bytes) -> str:
    """Die ersten zwölf Zeichen des SHA-256-Hashes."""
    raise NotImplementedError("TODO 9")

In [ ]:
# Selbsttest TODO 9
zeige(lambda: schreibe_csv(mitschreiben(lambda: importiere(EINGANG))[0][0]).decode("utf-8").splitlines()[:2])


def _t9_klein():
    braucht("TODO 3")
    probe = [Schicht("Illinois", dt.date(2026, 6, 29), "A", "Früh", 178, 165, 7, 12, 29.0, "IL_x.csv"),
             Schicht("Stammwerk", dt.date(2026, 7, 14), "3", "Spät", 249, 218, 4, 7, None, "stammwerk_x.csv")]
    daten = schreibe_csv(probe)
    assert isinstance(daten, bytes), f"schreibe_csv liefert {type(daten).__name__}, erwartet bytes"
    assert b"\r" not in daten, "Zeilenende \\r\\n: csv.writer braucht lineterminator=\"\\n\""
    assert not daten.startswith(b"\xef\xbb\xbf"), "UTF-8 ohne BOM"
    soll = ("werk,datum,linie,schicht,soll_stueck,ist_stueck,ausschuss_stueck,personal,hallentemp_c,quelle\n"
            "Illinois,2026-06-29,A,Früh,178,165,7,12,29.0,IL_x.csv\n"
            "Stammwerk,2026-07-14,3,Spät,249,218,4,7,,stammwerk_x.csv\n").encode("utf-8")
    assert daten == soll, f"erhalten {daten!r}"
    assert schreibe_csv([]) == soll.split(b"\n")[0] + b"\n", "eine leere Liste ergibt nur die Kopfzeile"


def _t9_pruefsumme():
    assert pruefsumme(b"") == "e3b0c44298fc", f"pruefsumme(b'') ist {pruefsumme(b'')!r}"
    assert pruefsumme(b"a\n") != pruefsumme(b"a\r\n") and len(pruefsumme(b"a\n")) == 12, "zwölf Zeichen, verschieden für \\n und \\r\\n"


def _t9_echt():
    braucht("TODO 8")
    schichten, _ = mitschreiben(lambda: importiere(EINGANG))[0]
    daten = schreibe_csv(schichten)
    zeilen_ = daten.decode("utf-8").split("\n")
    assert zeilen_[1] == "Illinois,2026-06-29,A,Früh,178,165,7,12,29.5,IL_shift_report_06292026.csv", f"erste Datenzeile {zeilen_[1]!r}"
    assert len(zeilen_) == 894 and zeilen_[-1] == "", f"{len(zeilen_) - 2} Datenzeilen, erwartet 892 und am Ende ein Zeilenumbruch"
    assert pruefsumme(daten) == "0bc618f824be", f"Prüfsumme {pruefsumme(daten)}, erwartet 0bc618f824be"


check("TODO 9 kleines Beispiel", _t9_klein)
check("TODO 9 Prüfsumme", _t9_pruefsumme)
check("TODO 9 schichten.csv aus allen Lieferungen", _t9_echt)

Die Prüfsumme ist `0bc618f824be`, im Browser, unter Windows, macOS und Linux. Sie stimmt nur, wenn alles festliegt: Reihenfolge, Zeilenende `\n`, Kodierung UTF-8 und Zahlformat. Mit `\r\n` als Zeilenende stünden in der Datei dieselben Werte, die Prüfsumme wäre trotzdem eine andere.

## E · Konfiguration und Scope
*Lerneinheiten 5 und 6 · Skript Abschnitte 6.2, 6.3 und 5.2*

### TODO 10 · Konfiguration mit Vorrang
Das Skript löst die Vorrangregel für die Log-Stufe allein. Hier gilt sie für alle drei Werte des Imports. `lade_konfig` ist die erste Zelle von `konfig.py`.

**Ziel:** `lade_konfig(toml_text, umgebung, argumente)` bestimmt `eingang`, `ausgang` und `log_level` nach der Vorrangregel.

**Signatur:** `lade_konfig(toml_text: str, umgebung: Mapping[str, str], argumente: Mapping[str, str | None]) -> dict[str, str]`

**Regeln**
- Für jeden Schlüssel aus `VOREINSTELLUNG` gilt der erste Wert dieser Treppe:
  1. das Argument `argumente[schluessel]`, wenn es den Schlüssel gibt und der Wert nicht `None` ist (`argparse` setzt `None`, wenn eine Option fehlt);
  2. die Umgebungsvariable `NORDWERK_EINGANG`, `NORDWERK_AUSGANG` oder `NORDWERK_LOG_LEVEL` aus `umgebung`;
  3. der Wert im Abschnitt `[import]` von `toml_text`, gelesen mit `tomllib.loads` (ein leerer Text ist eine leere Datei);
  4. die Voreinstellung aus `VOREINSTELLUNG`.
- Die Funktion liest nur ihre Parameter, keine Dateien und nicht `os.environ`. So lässt sie sich mit jedem Fall testen. Das Lesen übernimmt die vorgegebene Hülle in der Zelle danach.

In [ ]:
# → src/nordwerk/konfig.py
"""Konfiguration des Pakets nordwerk: Argument vor Umgebung vor nordwerk.toml vor Voreinstellung."""

import os
import tomllib
from collections.abc import Mapping
from pathlib import Path

VOREINSTELLUNG = {"eingang": "daten/roh", "ausgang": "daten/aufbereitet", "log_level": "WARNING"}


def lade_konfig(
    toml_text: str, umgebung: Mapping[str, str], argumente: Mapping[str, str | None]
) -> dict[str, str]:
    """Bestimmt eingang, ausgang und log_level nach der Vorrangregel."""
    raise NotImplementedError("TODO 10")

In [ ]:
# → src/nordwerk/konfig.py
def konfig_fuer_lauf(
    argumente: Mapping[str, str | None], datei: str = "nordwerk.toml"
) -> dict[str, str]:
    """Liest nordwerk.toml (falls vorhanden) und os.environ und wendet lade_konfig an."""
    pfad = Path(datei)
    toml_text = pfad.read_text(encoding="utf-8-sig") if pfad.is_file() else ""  # -sig: auch mit BOM
    return lade_konfig(toml_text, os.environ, argumente)

In [ ]:
# Selbsttest TODO 10
TOML_PROBE = '[import]\neingang = "lieferungen/roh"\nlog_level = "INFO"\n\n[bericht]\nausgang = "nicht/dieser"\n'
OHNE = {"eingang": None, "ausgang": None, "log_level": None}
zeige(lambda: lade_konfig(TOML_PROBE, {"NORDWERK_LOG_LEVEL": "ERROR"}, {**OHNE, "ausgang": "aus"}))


def _t10_faelle():
    vorher = os.environ.get("NORDWERK_LOG_LEVEL")
    os.environ["NORDWERK_LOG_LEVEL"] = "CRITICAL"
    try:
        faelle = [
            ("nur Voreinstellung", ("", {}, {}), {"eingang": "daten/roh", "ausgang": "daten/aufbereitet", "log_level": "WARNING"}),
            ("Argumente None", ("", {}, OHNE), {"eingang": "daten/roh", "ausgang": "daten/aufbereitet", "log_level": "WARNING"}),
            ("Datei", (TOML_PROBE, {}, OHNE), {"eingang": "lieferungen/roh", "ausgang": "daten/aufbereitet", "log_level": "INFO"}),
            ("Datei ohne [import]", ('[bericht]\neingang = "x"\n', {}, OHNE), {"eingang": "daten/roh", "ausgang": "daten/aufbereitet", "log_level": "WARNING"}),
            ("Umgebung vor Datei", (TOML_PROBE, {"NORDWERK_LOG_LEVEL": "ERROR", "NORDWERK_AUSGANG": "aus", "PATH": "/usr/bin"}, OHNE),
             {"eingang": "lieferungen/roh", "ausgang": "aus", "log_level": "ERROR"}),
            ("Argument vor Umgebung", (TOML_PROBE, {"NORDWERK_LOG_LEVEL": "ERROR"}, {"log_level": "DEBUG", "eingang": None}),
             {"eingang": "lieferungen/roh", "ausgang": "daten/aufbereitet", "log_level": "DEBUG"}),
        ]
        for name, argumente, soll in faelle:
            ist = lade_konfig(*argumente)
            assert ist == soll, f"{name}: {ist}, erwartet {soll}"
    finally:
        if vorher is None:
            del os.environ["NORDWERK_LOG_LEVEL"]
        else:
            os.environ["NORDWERK_LOG_LEVEL"] = vorher


check("TODO 10 Vorrang in sechs Fällen", _t10_faelle)

Was näher am einzelnen Lauf ist, gewinnt: Das Argument schlägt die Umgebungsvariable, die schlägt `nordwerk.toml`, die schlägt die Voreinstellung. Der Test setzt nebenbei `NORDWERK_LOG_LEVEL` in `os.environ` und stellt den alten Wert danach wieder her. Weil `lade_konfig` nur ihren Parameter `umgebung` liest, ändert das am Ergebnis nichts. Die Hülle `konfig_fuer_lauf` liest `nordwerk.toml` im aktuellen Ordner und `os.environ`; sie nutzt in TODO 12 die Kommandozeile.

### TODO 11 · Scope-Falle beheben
Das Skript zeigt die Falle an `notiere(warnung, gesammelt=[])`. Dieselbe Falle steckt in der vorgegebenen Funktion unten: Sie soll die abgelehnten Dateien eines Laufs sammeln.

**Ziel:** Korrigiere `merke_abgelehnt` so, dass jeder Aufruf ohne `liste` mit einer neuen, leeren Liste beginnt. Aufrufe bleiben gleich: `merke_abgelehnt(datei)` oder `merke_abgelehnt(datei, liste)`.

**Regeln**
- Die Voreinstellung `[]` legt Python genau einmal an, beim Ausführen von `def`. Alle Aufrufe ohne `liste` teilen sich dann dieselbe Liste.
- Nimm `None` als Voreinstellung und leg die Liste im Funktionsrumpf an. Typ: `list[str] | None`.
- Übergibt jemand eine eigene Liste, auch eine leere, wird genau diese ergänzt und zurückgegeben.

Führ die Zelle zuerst unverändert aus und schau dir an, was der erste Lauf am Ende enthält.

In [ ]:
# TODO 11: Diese Funktion hat eine Falle. Korrigiere sie.
def merke_abgelehnt(datei: str, liste: list[str] = []) -> list[str]:
    """Hängt datei an liste an und gibt die Liste zurück; ohne liste beginnt eine neue."""
    liste.append(datei)
    return liste


erster_lauf = merke_abgelehnt("stammwerk_2026-07-22.csv")
zweiter_lauf = merke_abgelehnt("IL_shift_report_08102026.csv")
print("erster Lauf: ", erster_lauf)
print("zweiter Lauf:", zweiter_lauf)

In [ ]:
# Selbsttest TODO 11
def _t11():
    assert zweiter_lauf == ["IL_shift_report_08102026.csv"], f"zweiter Lauf {zweiter_lauf}: Er enthält die Datei des ersten Laufs mit"
    a = merke_abgelehnt("a.csv")
    b = merke_abgelehnt("b.csv")
    assert b == ["b.csv"] and a is not b, f"zweiter Aufruf liefert {b}: Die Voreinstellung wird geteilt"
    meine = []
    zurueck = merke_abgelehnt("x.csv", meine)
    assert zurueck is meine and meine == ["x.csv"], f"eine übergebene leere Liste wurde nicht ergänzt ({meine}); Vorsicht mit 'liste or []'"
    standard = inspect.signature(merke_abgelehnt).parameters["liste"].default
    assert not isinstance(standard, list), f"die Voreinstellung ist noch eine Liste: {standard!r}"


check("TODO 11 jeder Lauf beginnt leer", _t11)

Mit der Falle enthielt der erste Lauf am Ende beide Dateien: `erster_lauf` und `zweiter_lauf` sind dieselbe Liste. Im Notebook fällt das beim zweiten Ausführen einer Zelle auf, in einem Programm erst, wenn ein Lauf Ergebnisse eines früheren mitzählt. ruff findet diese Falle: Die Regel B006 aus der Gruppe `B`, die in `pyproject.toml` eingeschaltet ist, meldet jede veränderliche Voreinstellung. Vorsicht mit der Abkürzung `liste = liste or []`: Eine übergebene leere Liste gilt als falsch und wird durch eine neue ersetzt.

## F · Kommandozeile
*Lerneinheit 3 · Skript Abschnitte 3.3 bis 3.6*

### TODO 12 · Kommandozeile
Das Skript baut den Unterbefehl `nordwerk info`. Hier baust du `nordwerk import`, den Befehl, der am Ende auf deinem Laptop und im frischen Klon die Prüfsumme zeigt.

**Ziel:** `main(argv)` liest die Argumente, führt den Import aus, schreibt beide Dateien und gibt den Exit-Code zurück.

**Signatur:** `main(argv: list[str] | None = None) -> int`

**Regeln**
- `argparse.ArgumentParser(prog="nordwerk", …)` mit `--version` (Ausgabe `nordwerk 0.1.0`, aus `__version__`) und dem Unterbefehl `import`. Er hat das optionale Argument `eingang` (`nargs="?"`) und die Optionen `--aus` und `--log-level`. Keine Voreinstellungen in `argparse`: Fehlt etwas, ist der Wert `None`, und die Konfiguration entscheidet.
- Konfiguration mit `konfig.konfig_fuer_lauf({"eingang": …, "ausgang": …, "log_level": …})`, Logging mit `richte_logging_ein(…)`.
- Gibt es in `eingang` keine Lieferdateien (`importer.lieferungen`) oder fehlt der Ordner, schreib eine Meldung nach `sys.stderr` und gib 1 zurück. Eine unbekannte Log-Stufe oder eine `nordwerk.toml`, die kein gültiges TOML ist (`tomllib.TOMLDecodeError`), ergibt ebenso eine Meldung und 2. Kein Traceback für erwartbare Fehler.
- Sonst `importer.importiere`, `importer.schreibe_csv` und `importer.pruefsumme`. Die Prüfsumme kommt als letzter Schlüssel `"pruefsumme"` in den Bericht. Den Ordner `ausgang` anlegen (`mkdir(parents=True, exist_ok=True)`), dorthin `schichten.csv` (die Bytes aus `schreibe_csv`) und `qualitaet.json` schreiben: `json.dumps(bericht, ensure_ascii=False, indent=2)` mit Zeilenumbruch am Ende, als UTF-8.
- Zum Schluss genau diese drei Zeilen auf stdout und Rückgabe 0:
```
Dateien: 70 gelesen, 2 abgelehnt
Zeilen: 892 übernommen, 4 verworfen, 12 Dubletten
Prüfsumme: 0bc618f824be
```
- `main` **gibt** den Exit-Code **zurück** und ruft nie `sys.exit` auf. Sonst löst jeder Aufruf im Notebook oder im Test `SystemExit` aus, statt eine Zahl zu liefern, die du prüfen kannst.

In [ ]:
# → src/nordwerk/cli.py
def main(argv: list[str] | None = None) -> int:
    """Kommandozeile: nordwerk import [EINGANG] [--aus AUSGANG] [--log-level STUFE]."""
    raise NotImplementedError("TODO 12")

In [ ]:
# Selbsttest TODO 12
ZUSAMMENFASSUNG = "Dateien: 70 gelesen, 2 abgelehnt\nZeilen: 892 übernommen, 4 verworfen, 12 Dubletten\nPrüfsumme: 0bc618f824be\n"


def _aufruf(argv):
    """Ruft main(argv) auf und liefert (Rückgabe, stdout, stderr)."""
    aus, fehler = io.StringIO(), io.StringIO()
    with logger_gesichert(), contextlib.redirect_stdout(aus), contextlib.redirect_stderr(fehler):
        rueckgabe = main(argv)
    return rueckgabe, aus.getvalue(), fehler.getvalue()


def _t12_lauf():
    braucht("TODO 7", "TODO 8", "TODO 9", "TODO 10")
    with tempfile.TemporaryDirectory() as ordner:
        rueckgabe, aus, fehler = _aufruf(["import", str(EINGANG), "--aus", ordner, "--log-level", "WARNING"])
        assert rueckgabe == 0, f"Rückgabe {rueckgabe!r}, erwartet 0"
        assert aus == ZUSAMMENFASSUNG, f"Zusammenfassung {aus!r}"
        stufen = sorted(z.split(" ", 1)[0] for z in fehler.splitlines())
        assert stufen == ["ERROR"] * 2 + ["WARNING"] * 4, f"auf stderr {stufen}, erwartet 4 Warnungen und 2 Fehler"
        daten = (Path(ordner) / "schichten.csv").read_bytes()
        assert pruefsumme(daten) == "0bc618f824be", f"schichten.csv hat die Prüfsumme {pruefsumme(daten)}"
        text = (Path(ordner) / "qualitaet.json").read_text(encoding="utf-8")
        bericht = json.loads(text)
        assert list(bericht) == ["dateien", "kodierung", "zeilen", "werke", "pruefsumme"], f"Schlüssel {list(bericht)}"
        assert bericht["pruefsumme"] == "0bc618f824be" and text.startswith('{\n  "dateien"') and text.endswith("}\n"), "qualitaet.json mit indent=2, Prüfsumme und Zeilenumbruch am Ende"


def _t12_leer():
    braucht("TODO 7", "TODO 8", "TODO 9", "TODO 10")
    with tempfile.TemporaryDirectory() as ordner:
        for eingang in [ordner, str(Path(ordner) / "gibt_es_nicht")]:
            rueckgabe, aus, fehler = _aufruf(["import", eingang, "--aus", str(Path(ordner) / "aus")])
            assert rueckgabe == 1, f"ohne Lieferungen Rückgabe {rueckgabe!r}, erwartet 1"
            assert aus == "" and fehler.strip() and "Traceback" not in fehler, "die Meldung gehört nach stderr, ohne Traceback"
            assert not (Path(ordner) / "aus").exists(), "ohne Lieferungen nichts schreiben"


def _t12_konfig():
    braucht("TODO 7", "TODO 8", "TODO 9", "TODO 10")
    namen = ("NORDWERK_EINGANG", "NORDWERK_AUSGANG", "NORDWERK_LOG_LEVEL")
    vorher = {n: os.environ.get(n) for n in namen}
    with tempfile.TemporaryDirectory() as ordner:
        os.environ.update({"NORDWERK_EINGANG": str(EINGANG), "NORDWERK_AUSGANG": ordner, "NORDWERK_LOG_LEVEL": "INFO"})
        try:
            rueckgabe, aus, fehler = _aufruf(["import", "--log-level", "ERROR"])
        finally:
            for n, wert in vorher.items():
                if wert is None:
                    os.environ.pop(n, None)
                else:
                    os.environ[n] = wert
        assert rueckgabe == 0 and (Path(ordner) / "schichten.csv").is_file(), "Eingang und Ausgang kommen ohne Argumente aus NORDWERK_EINGANG und NORDWERK_AUSGANG"
        assert sorted(z.split(" ", 1)[0] for z in fehler.splitlines()) == ["ERROR", "ERROR"], "--log-level ERROR schlägt NORDWERK_LOG_LEVEL=INFO"
    aus = io.StringIO()
    try:
        with contextlib.redirect_stdout(aus):
            main(["--version"])
    except SystemExit as e:
        assert e.code == 0 and aus.getvalue().strip() == "nordwerk 0.1.0", f"--version zeigt {aus.getvalue().strip()!r}"
    else:
        raise AssertionError("--version fehlt")


check("TODO 12 nordwerk import", _t12_lauf)
check("TODO 12 ohne Lieferungen Rückgabe 1", _t12_leer)
check("TODO 12 Konfiguration und --version", _t12_konfig)

Jetzt läuft der Import wie auf der Kommandozeile, nur dass die Wörter als Liste kommen. Im Browser landen die Dateien unter `daten/aufbereitet/`, in VS Code unter `notebooks/daten/aufbereitet/`. Die Warnungen und Fehler stehen über der Zusammenfassung.

In [ ]:
# Wie auf der Kommandozeile: nordwerk import daten/roh --aus daten/aufbereitet
def _import_lauf():
    rueckgabe = main(["import", "daten/roh", "--aus", "daten/aufbereitet"])
    print("Rückgabe:", rueckgabe, "| in daten/aufbereitet:", sorted(p.name for p in Path("daten/aufbereitet").iterdir()))


zeige(_import_lauf)

### Der Startblock von `cli.py`
Im Paket endet `cli.py` mit dem Block aus Skript Abschnitt 3.4. Er startet `main`, wenn du das Modul mit `python -m nordwerk.cli` aufrufst, etwa im Debugger, und bleibt beim Import still. Im Notebook heißt der Namensraum `"__main__"`, der Block liefe also los. Die Zelle davor gibt dem Namensraum deshalb kurz den Namen, den das Modul im Paket hat, die Zelle danach stellt ihn zurück. Beide gehören nicht ins Repository.

In [ ]:
# Nur im Notebook: Im Paket heißt dieses Modul nordwerk.cli.
__name__ = "nordwerk.cli"

In [ ]:
# → src/nordwerk/cli.py
if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
# Nur im Notebook: wieder der gewohnte Name.
__name__ = "__main__"
print("Der Startblock ist nicht gelaufen. Er läuft nur mit python -m nordwerk.cli.")

## Bonus
### Bonus 1 · Warum Generator: Speicher messen
Das Skript überlegt, warum ein Generator weniger Speicher braucht als eine Liste. Hier misst du es mit `tracemalloc` aus der Standardbibliothek: `tracemalloc.start()` beginnt die Messung, `tracemalloc.get_traced_memory()` liefert `(aktuell, spitze)` in Byte, `tracemalloc.stop()` beendet sie.

Die Zelle unten gibt dir `alle_zeilen(eingang)`: einen Generator, der mit `yield from` die Generatoren aller lesbaren Lieferungen hintereinanderschaltet. `ruhig()` schaltet die Meldungen während der Messung ab.

1. `spitze(arbeit)`: führt `arbeit()` aus und gibt den Spitzenwert des Speichers in Byte zurück. Beende die Messung auch dann, wenn `arbeit()` eine Ausnahme auslöst (`try` und `finally`).
2. `spitze_liste`: der Spitzenwert, während `list(alle_zeilen(EINGANG))` entsteht.
3. `spitze_generator`: der Spitzenwert, während eine Schleife `alle_zeilen(EINGANG)` nur durchläuft und zählt.

In [ ]:
def alle_zeilen(eingang):
    """Alle gültigen Schichten aller lesbaren Lieferungen, eine nach der anderen (Dubletten bleiben drin).

    Nur für die Messung: Eine Datei, die sich nicht lesen lässt, wird übersprungen."""
    for werk, pfad in lieferungen(eingang):
        try:
            yield from zeilen(pfad, werk, {})
        except (LieferungsFehler, ValueError):
            pass


@contextlib.contextmanager
def ruhig():
    """Während des Blocks schreibt der Logger "nordwerk" nichts."""
    with logger_gesichert() as logger:
        logger.setLevel(logging.CRITICAL + 10)
        yield


# Ein Durchlauf vorab füllt die Zwischenspeicher von Python (etwa für strptime), damit sie keine Messung verfälschen.
with ruhig():
    zeige(lambda: print("Vorab durchlaufen:", sum(1 for _ in alle_zeilen(EINGANG)), "Schichten"))

In [ ]:
# Bonus 1
import tracemalloc


def spitze(arbeit):
    """Führt arbeit() aus und gibt den Spitzenwert des Speichers in Byte zurück."""
    raise NotImplementedError("Bonus 1")


with ruhig():
    spitze_liste = None
    spitze_generator = None
print(spitze_liste, spitze_generator)

In [ ]:
# Selbsttest Bonus 1
def _b1():
    braucht("TODO 1", "TODO 2", "TODO 3", "TODO 4", "TODO 5", "TODO 6")
    groesse = spitze(lambda: bytearray(400_000))
    assert isinstance(groesse, int) and 400_000 <= groesse < 600_000, f"spitze(lambda: bytearray(400_000)) ergibt {groesse!r}, erwartet etwas über 400.000"
    assert not tracemalloc.is_tracing(), "die Messung läuft nach spitze() noch: tracemalloc.stop() fehlt"
    try:
        spitze(lambda: 1 / 0)
    except ZeroDivisionError:
        pass
    assert not tracemalloc.is_tracing(), "nach einer Ausnahme läuft die Messung weiter: stop() gehört in finally"
    assert isinstance(spitze_liste, int) and isinstance(spitze_generator, int), "spitze_liste und spitze_generator fehlen"
    assert spitze_liste > 100_000, f"spitze_liste {spitze_liste} Byte: Die Liste aller Schichten braucht mehr"
    assert spitze_liste > 3 * spitze_generator, f"Liste {spitze_liste} Byte, Generator {spitze_generator} Byte: erwartet ein Vielfaches"


check("Bonus 1 Liste gegen Generator", _b1)
if isinstance(spitze_liste, int) and isinstance(spitze_generator, int) and spitze_generator > 0:
    print(f"Liste: {spitze_liste / 1000:.0f} kB, Generator: {spitze_generator / 1000:.0f} kB, Faktor {spitze_liste / spitze_generator:.1f}")

Die Liste hält alle Schichten gleichzeitig, der Generator immer nur eine Datei. Die genauen Zahlen hängen von der Python-Version ab, das Verhältnis bleibt. Bei den 72 kleinen Dateien sind beide Werte klein; bei Lieferungen mit Millionen Zeilen entscheidet genau dieser Unterschied, ob ein Programm läuft.

### Bonus 2 · `.env` von Hand lesen
uv und VS Code lesen `.env` für dich, das Paket sieht nur `os.environ` (Skript Abschnitt 6.4). Wie schwer ist das Lesen selbst?

**Ziel:** `lies_env(text)` liest den Inhalt einer `.env`-Datei in ein Wörterbuch.

**Regeln**
- Je Zeile ein Paar `NAME=wert`. Leerraum um Name und Wert entfernen, am ersten `=` trennen: Ein Wert darf selbst `=` enthalten.
- Leere Zeilen und Zeilen, die (nach Leerraum) mit `#` beginnen, überspringen. Zeilen ohne `=` ebenso.
- Steht der Wert in passenden Anführungszeichen, `"…"` oder `'…'`, gehören sie nicht zum Wert.
- Die Datei kann mit Windows-Zeilenende `\r\n` kommen.

In [ ]:
# Bonus 2
def lies_env(text):
    """Liest NAME=wert-Zeilen einer .env-Datei in ein Wörterbuch."""
    raise NotImplementedError("Bonus 2")


zeige(lambda: lies_env("# Probe\nNORDWERK_LOG_LEVEL=INFO\n"))

In [ ]:
# Selbsttest Bonus 2
def _b2():
    text = ("# Nur auf diesem Laptop, nie ins Repository\r\n"
            "NORDWERK_LOG_LEVEL=INFO\r\n"
            "\r\n"
            "  NORDWERK_AUSGANG = \"daten/meine ausgabe\"  \r\n"
            "TOOLBOX_KEY='sk-probe=123'\r\n"
            "    # eingerückter Kommentar\r\n"
            "LEER=\r\n"
            "ZITAT=\"halb'\r\n"
            "kaputte Zeile\r\n")
    soll = {"NORDWERK_LOG_LEVEL": "INFO", "NORDWERK_AUSGANG": "daten/meine ausgabe", "TOOLBOX_KEY": "sk-probe=123", "LEER": "", "ZITAT": "\"halb'"}
    ist = lies_env(text)
    assert ist == soll, f"erhalten {ist}"
    assert lies_env("") == {}, "ein leerer Text ergibt ein leeres Wörterbuch"


check("Bonus 2 .env lesen", _b2)

Schon diese wenigen Regeln brauchen einige Fälle. Echte `.env`-Dateien kennen noch mehr: `export` vor dem Namen, Zeilenumbrüche in Anführungszeichen, Verweise auf andere Variablen. Deshalb liest das Paket nur `os.environ` und überlässt `.env` den Werkzeugen, die das schon können: `uv run --env-file .env` und dem Debugger von VS Code.

## Ins Repo
Jetzt wandert der Code in dein Repository. Wie du es anlegst und einrichtest, steht im Schritt **„Auf deinem Laptop“** der Kursseite. Übernimm jede Zelle, die mit `# → src/nordwerk/…` beginnt, ohne diese erste Zeile und in der Reihenfolge des Notebooks, in die genannte Datei. Zwischen zwei Zellen lässt du zwei Leerzeilen.

| Zellen im Notebook | Datei im Paket |
|---|---|
| Vorgaben (Importe, `log`, `LieferungsFehler`, `ZeilenFehler`, `GRUENDE`, `Format`, `FORMATE`, `SCHICHT_RANG`, `PFLICHT`), TODO 1 bis 6, TODO 8 und 9 | `src/nordwerk/importer.py` |
| TODO 10 (`lade_konfig` mit Importen und `VOREINSTELLUNG`) und die Hülle `konfig_fuer_lauf` | `src/nordwerk/konfig.py` |
| TODO 7 (`richte_logging_ein` mit Importen), TODO 12 (`main`) und der Startblock `if __name__ == "__main__":` | `src/nordwerk/cli.py` |
| keine Zelle: nur `__version__ = "0.1.0"` | `src/nordwerk/__init__.py` |

Nicht ins Paket gehören die Hilfen für die Selbsttests, die Datenzelle, die Selbsttests selbst, die Zellen „Nur im Notebook“, die Übung zu TODO 11 und die Bonusaufgaben. In `pyproject.toml` zeigt der Befehl auf die neue Funktion: `nordwerk = "nordwerk.cli:main"` unter `[project.scripts]`. Danach prüfst du mit `uv run ruff format`, `uv run ruff check` und `uv run mypy src`, dass alles sauber ist.

**Nachweis:** `uv run nordwerk import` zeigt auf deinem Laptop dieselbe Prüfsumme `0bc618f824be` wie dieses Notebook. Dasselbe gilt in einem frischen Klon deines Repositorys nach `uv sync --locked`: Damit ist belegt, dass dein Code auf jedem Rechner gleich läuft.

## Zum Nachdenken

1. Warum lehnt der Importer die Datei ohne `Scrap Qty` ganz ab, verwirft bei fehlender Ist-Stückzahl aber nur die eine Zeile?
2. `stammwerk_2026-08-06.csv` enthält die zwölf Zeilen vom 5. August ein zweites Mal. Woran würdest du merken, wenn die Werte darin von der ersten Lieferung abwichen, und was sollte der Importer dann tun?
3. Die Prüfsumme hängt an der Reihenfolge der Zeilen. Warum sortiert `importiere` am Ende selbst, obwohl `lieferungen` die Dateien schon sortiert liefert?

<details><summary>Hinweise zu den Antworten</summary>

1. Ohne die Spalte fehlt der Ausschuss in jeder Zeile der Datei, keine Zeile wäre vollständig. Eine einzelne leere Zelle betrifft nur ihre Zeile, der Rest der Datei ist brauchbar. Datei- und Zeilenfehler sind deshalb verschiedene Ausnahmen (Skript Abschnitt 4.4).
2. So wie er gebaut ist, gar nicht: Er behält die zuerst gelesene Zeile und zählt die zweite als Dublette. Ein strengerer Importer vergleicht die Werte und meldet Abweichungen mit `log.warning` oder lehnt die Datei ab. Welche Regel gilt, klärst du mit den Werken, nicht im Code allein.
3. Die Dateien eines Werks sind nach Namen sortiert, die Zeilen in einer Datei stehen aber in der Reihenfolge, die das Werk schreibt. Das Stammwerk schreibt erst alle vier Linien der Frühschicht, dann die der Spätschicht und der Nachtschicht; die Tabelle ordnet nach Linie und dann nach Schicht. Ändert ein Werk seinen Export, bliebe die Prüfsumme nur mit der eigenen Sortierung gleich. Erst sie legt die Reihenfolge unabhängig von den Lieferungen fest.
</details>

## Wie es weitergeht

Du hast Lieferdateien sortiert gefunden, drei Kodierungen sicher gelesen, Schichtmeldungen als Datentyp gebaut und jede Zeile mit eigenem Grund geprüft. Ein Generator liefert die Schichten Datei für Datei, das Logging zeigt jede Meldung genau einmal, und `importiere` schreibt einen Bericht, dessen Bilanz aufgeht. `schreibe_csv` legt die Tabelle Byte für Byte fest, `lade_konfig` folgt der Vorrangregel, und `main` macht daraus den Befehl `nordwerk import`. In **Modul 1 „Git, Tests und Teamarbeit“** prüfen Tests und eine automatische Prüfung (CI) diesen Importer bei jeder Änderung. Die Leitfrage dort: Wie arbeiten zwei Personen am selben Code, ohne dass etwas verloren geht oder unbemerkt kaputtgeht?